# Experiment 11: Model Optimization Experiment

## Objective

To optimize a pre-trained language model using quantization
and compare the original model with the optimized model.

## Technique

Quantization

## Workflow

Pre-trained Model
      ↓
Measure Model Size
      ↓
Apply Quantization
      ↓
Optimized Model
      ↓
Compare Model Size
      ↓
Compare Inference

In [1]:
print("Model Optimization Experiment")
print("Experiment 11")

Model Optimization Experiment
Experiment 11


In [2]:
!pip -q install transformers sentencepiece

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Pre-trained model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Pre-trained model loaded successfully!


In [4]:
original_size = sum(
    param.numel() * param.element_size()
    for param in model.parameters()
)

original_size_mb = original_size / (1024 ** 2)

print(f"Original model size: {original_size_mb:.2f} MB")

Original model size: 293.58 MB


In [7]:
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Quantization completed successfully!")

/tmp/ipykernel_1282/1322333341.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Quantization completed successfully!


In [8]:
quantized_size = sum(
    param.numel() * param.element_size()
    for param in quantized_model.parameters()
)

quantized_size_mb = quantized_size / (1024 ** 2)

print(f"Original model size: {original_size_mb:.2f} MB")
print(f"Quantized model size: {quantized_size_mb:.2f} MB")

Original model size: 293.58 MB
Quantized model size: 62.83 MB


In [9]:
import time

prompt = "What is artificial intelligence?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Original model
start_time = time.time()

original_output = model.generate(
    **inputs,
    max_new_tokens=30
)

original_time = time.time() - start_time

# Quantized model
start_time = time.time()

quantized_output = quantized_model.generate(
    **inputs,
    max_new_tokens=30
)

quantized_time = time.time() - start_time

original_answer = tokenizer.decode(
    original_output[0],
    skip_special_tokens=True
)

quantized_answer = tokenizer.decode(
    quantized_output[0],
    skip_special_tokens=True
)

print("Original Model Answer:")
print(original_answer)

print("\nQuantized Model Answer:")
print(quantized_answer)

print("\nOriginal Model Time:", round(original_time, 4), "seconds")
print("Quantized Model Time:", round(quantized_time, 4), "seconds")

Original Model Answer:
artificial intelligence

Quantized Model Answer:
artificial intelligence

Original Model Time: 0.3661 seconds
Quantized Model Time: 0.2321 seconds


In [10]:
size_reduction = (
    (original_size_mb - quantized_size_mb)
    / original_size_mb
) * 100

print(f"Original Model Size: {original_size_mb:.2f} MB")
print(f"Quantized Model Size: {quantized_size_mb:.2f} MB")
print(f"Size Reduction: {size_reduction:.2f}%")

Original Model Size: 293.58 MB
Quantized Model Size: 62.83 MB
Size Reduction: 78.60%


In [11]:
import pandas as pd

optimization_results = {
    "Metric": [
        "Original Model Size (MB)",
        "Quantized Model Size (MB)",
        "Size Reduction (%)",
        "Original Inference Time (sec)",
        "Quantized Inference Time (sec)"
    ],
    "Result": [
        round(original_size_mb, 2),
        round(quantized_size_mb, 2),
        round(size_reduction, 2),
        round(original_time, 4),
        round(quantized_time, 4)
    ]
}

optimization_df = pd.DataFrame(optimization_results)

display(optimization_df)

,Metric,Result
0,Original Model Size (MB),293.5800
1,Quantized Model Size (MB),62.8300
2,Size Reduction (%),78.6000
3,Original Inference Time (sec),0.3661
4,Quantized Inference Time (sec),0.2321


# Experiment 11: Model Optimization Experiment

## Objective

To optimize a pre-trained language model using quantization and
compare the original model with the optimized model.

## Model

Pre-trained model: google/flan-t5-small

## Optimization Technique

Dynamic INT8 Quantization

## Process

1. Loaded the pre-trained model.
2. Measured the original model size.
3. Applied dynamic quantization to Linear layers.
4. Measured the quantized model size.
5. Compared inference performance.
6. Calculated the size reduction.

## Results

The original and quantized models were compared based on:

- Model size
- Size reduction
- Inference time
- Generated output

## Conclusion

Quantization is a model optimization technique that converts
model parameters or computations to lower-precision representations.

In this experiment, dynamic INT8 quantization was applied to the
FLAN-T5-small model. The optimized model was then compared with
the original model to observe changes in model size and inference
performance.